# SO(3) benchmarking and profiling

Runs the generic benchmark helper with an explicit SO(3) backend, then profiles the same benchmark call with the standard-library `cProfile` profiler.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

start = Path.cwd().resolve()
candidates = [start, *start.parents]
try:
    candidates.extend(path for path in start.iterdir() if path.is_dir())
except OSError:
    pass

PROJECT_ROOT = next(
    (
        path
        for path in candidates
        if (path / "pyproject.toml").exists()
        and (path / "src" / "equivariant_polynomials").is_dir()
        and (path / "benchmarks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not find the project root. Start Jupyter from the project directory "
        "or from its notebooks directory."
    )

for import_path in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    import_path = str(import_path)
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Users\micha\OneDrive\Desktop\Oden\Notes\Equivariant Polynomial Tensor Functions\Python\equivariant_polynomials


In [2]:
import cProfile
import io
import pstats

from pprint import pp

from benchmarks import benchmark
from equivariant_polynomials.groups.so3 import SO3RepresentationTheory, hilbert_series_so3, hilbert_series_so3_multigraded


In [3]:
random_seed = 12345
input_irreps = (2, 3)
input_multiplicities = (2, 1)
output_irreps = (4,)
max_degree = 9
modulus = 2521
profile_sort = "cumtime"
profile_rows = 25

benchmark_config = {
    "hilbert_series": hilbert_series_so3,
    "hilbert_series_multigraded": hilbert_series_so3_multigraded,
    "input_irreps": input_irreps,
    "input_multiplicities": input_multiplicities,
    "max_degree": max_degree,
    "random_seed": random_seed,
    "modulus": modulus,
}


## Benchmark


In [4]:
theory = SO3RepresentationTheory()
summaries = []

for output_irrep in output_irreps:
    print(f"\nRun: output_irrep={output_irrep}, max_degree={max_degree}")
    summaries.append(
        benchmark(
            theory=theory,
            output_irrep=output_irrep,
            **benchmark_config,
        )
    )

summaries = tuple(summaries)
pp(summaries)



Run: output_irrep=4, max_degree=9
algebra generators: 2.330 s
module generators: 7.270 s
total: 9.626 s
({'scenario': {'input_irreps': (2, 3),
               'input_multiplicities': (2, 1),
               'output_irrep': 4,
               'max_degree': 9,
               'random_seed': 12345,
               'modulus': 2521},
  'algebra': {'hilbert_dimensions': (1, 0, 4, 7, 24, 54, 156, 340, 817, 1739),
              'generator_counts': (0, 0, 4, 7, 14, 26, 52, 68, 60, 39),
              'matches_known_generators': True,
              'generator_seconds': 2.32979750004597},
  'module': {'hilbert_dimensions': (0,
                                    0,
                                    6,
                                    21,
                                    87,
                                    273,
                                    807,
                                    2109,
                                    5205,
                                    11919),
             

## cProfile


In [5]:
profiled_output_irrep = output_irreps[0]
profiler = cProfile.Profile()
profile_summary = profiler.runcall(
    benchmark,
    theory=SO3RepresentationTheory(),
    output_irrep=profiled_output_irrep,
    **benchmark_config,
)

stats_stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stats_stream)
stats.strip_dirs().sort_stats(profile_sort).print_stats(profile_rows)
print(stats_stream.getvalue())

profile_summary


algebra generators: 2.771 s
module generators: 9.194 s
total: 12.041 s
         3450651 function calls (3438350 primitive calls) in 11.984 seconds

   Ordered by: cumulative time
   List reduced from 392 to 25 due to restriction <25>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        7    0.000    0.000   15.131    2.162 base_events.py:1977(_run_once)
        2    0.010    0.005    9.113    4.557 benchmark.py:73(run_generators)
        2    0.018    0.009    9.035    4.518 generators.py:105(extract_independent_generators)
      248    0.017    0.000    6.630    0.027 generators.py:403(_stream_content_generators)
        1    0.001    0.001    6.342    6.342 benchmark.py:57(benchmark)
        6    0.001    0.000    5.777    0.963 selectors.py:310(select)
      917    0.004    0.000    5.525    0.006 generators.py:504(_syndrome_batches)
      421    0.198    0.000    5.003    0.012 generators.py:360(_previous_content_product_basis)
      707    0.027    0.00

{'scenario': {'input_irreps': (2, 3),
  'input_multiplicities': (2, 1),
  'output_irrep': 4,
  'max_degree': 9,
  'random_seed': 12345,
  'modulus': 2521},
 'algebra': {'hilbert_dimensions': (1, 0, 4, 7, 24, 54, 156, 340, 817, 1739),
  'generator_counts': (0, 0, 4, 7, 14, 26, 52, 68, 60, 39),
  'matches_known_generators': True,
  'generator_seconds': 2.771434099995531},
 'module': {'hilbert_dimensions': (0,
   0,
   6,
   21,
   87,
   273,
   807,
   2109,
   5205,
   11919),
  'generator_counts': (0, 0, 6, 21, 63, 147, 264, 284, 133, 35),
  'matches_known_generators': True,
  'generator_seconds': 9.193762900074944},
 'total_seconds': 12.04131130001042}